In [1]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import pandas as pd
import matplotlib.pyplot as plt

# Carga de datos
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Procesamiento de df_train
# LLENAR DATOS VACIOS
df_train.loc[df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_train.loc[df_train['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_train.loc[df_train['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_train.loc[df_train['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

# Corregir caracteres especiales
df_train['ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_PRGM_DEPARTAMENTO'] = df_train['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_train['ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_train.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

# Eliminar columnas que no sirven
df_train.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)


df_train['frecuencia_ESTU_VALORMATRICULAUNIVERSIDAD'] = df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].map(df_train['ESTU_VALORMATRICULAUNIVERSIDAD'].value_counts())
df_train.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

df_train['frecuencia_ESTU_PRGM_DEPARTAMENTO'] = df_train['ESTU_PRGM_DEPARTAMENTO'].map(df_train['ESTU_PRGM_DEPARTAMENTO'].value_counts())
df_train.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

df_train['frecuencia_ESTU_HORASSEMANATRABAJA'] = df_train['ESTU_HORASSEMANATRABAJA'].map(df_train['ESTU_HORASSEMANATRABAJA'].value_counts())
df_train.drop(columns=['ESTU_HORASSEMANATRABAJA'], inplace=True)

df_train['frecuencia_ESTU_PRGM_ACADEMICO'] = df_train['ESTU_PRGM_ACADEMICO'].map(df_train['ESTU_PRGM_ACADEMICO'].value_counts())
df_train.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)


# Mapeos
mapeo_estrato = {
    'Sin Estrato': 0,
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6
}
mapeo_si_no = {
    'No': 0,
    'Si': 1,
}
df_train['FAMI_ESTRATOVIVIENDA'] = df_train['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_train['FAMI_TIENEINTERNET'] = df_train['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_train['ESTU_PAGOMATRICULAPROPIO'] = df_train['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_train['FAMI_TIENECOMPUTADOR'] = df_train['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)


# Convertir RENDIMIENTO_GLOBAL a categórico
dummies = df_train['RENDIMIENTO_GLOBAL'].str.get_dummies()
df_train = pd.concat([df_train, dummies], axis=1)
df_train.drop(columns=['RENDIMIENTO_GLOBAL'], inplace=True)

df_train.to_csv('train_modified.csv', index=False)



# Procesamiento de df_test
df_test.loc[df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df_test.loc[df_test['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df_test.loc[df_test['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df_test.loc[df_test['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'

df_test['ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_PRGM_DEPARTAMENTO'] = df_test['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_VALORMATRICULAUNIVERSIDAD'] = df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df_test['ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


df_test.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

df_test.drop(columns=['FAMI_TIENELAVADORA', 'FAMI_TIENEAUTOMOVIL', 'ESTU_PRIVADO_LIBERTAD', 'FAMI_EDUCACIONMADRE', 'FAMI_EDUCACIONPADRE'], inplace=True)

df_test['frecuencia_ESTU_VALORMATRICULAUNIVERSIDAD'] = df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].map(df_test['ESTU_VALORMATRICULAUNIVERSIDAD'].value_counts())
df_test.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)

df_test['frecuencia_ESTU_PRGM_DEPARTAMENTO'] = df_test['ESTU_PRGM_DEPARTAMENTO'].map(df_test['ESTU_PRGM_DEPARTAMENTO'].value_counts())
df_test.drop(columns=['ESTU_PRGM_DEPARTAMENTO'], inplace=True)

df_test['frecuencia_ESTU_HORASSEMANATRABAJA'] = df_test['ESTU_HORASSEMANATRABAJA'].map(df_test['ESTU_HORASSEMANATRABAJA'].value_counts())
df_test.drop(columns=['ESTU_HORASSEMANATRABAJA'], inplace=True)

df_test['frecuencia_ESTU_PRGM_ACADEMICO'] = df_test['ESTU_PRGM_ACADEMICO'].map(df_test['ESTU_PRGM_ACADEMICO'].value_counts())
df_test.drop(columns=['ESTU_PRGM_ACADEMICO'], inplace=True)



df_test['FAMI_ESTRATOVIVIENDA'] = df_test['FAMI_ESTRATOVIVIENDA'].map(mapeo_estrato)
df_test['FAMI_TIENEINTERNET'] = df_test['FAMI_TIENEINTERNET'].map(mapeo_si_no)
df_test['ESTU_PAGOMATRICULAPROPIO'] = df_test['ESTU_PAGOMATRICULAPROPIO'].map(mapeo_si_no)
df_test['FAMI_TIENECOMPUTADOR'] = df_test['FAMI_TIENECOMPUTADOR'].map(mapeo_si_no)


df_test.to_csv('test_modified.csv', index=False)


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import randint
import numpy as np

# Carga de datos limpios
df_train = pd.read_csv('train_modified.csv')
df_test = pd.read_csv('test_modified.csv')

# Separar características y etiquetas en el conjunto de entrenamiento
X_train = df_train.drop(columns=['bajo', 'medio-bajo', 'medio-alto', 'alto'])
y_train = df_train[['bajo', 'medio-bajo', 'medio-alto', 'alto']]

# Convertir etiquetas a una sola columna de categorías
y_train = y_train.idxmax(axis=1)

# Para el conjunto de test, mantener solo las características
X_test = df_test.copy()

# Imputación de valores faltantes
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Crear un pipeline que incluye escalado y el modelo de árbol de decisión
pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('dt', DecisionTreeClassifier(random_state=42))
])

# Definir la distribución de parámetros para la búsqueda aleatoria
param_distributions = {
    'dt__max_depth': randint(1, 20),
    'dt__min_samples_split': randint(2, 20),
    'dt__min_samples_leaf': randint(1, 20)
}

# Realizar la búsqueda aleatoria de hiperparámetros con validación cruzada
random_search = RandomizedSearchCV(estimator=pipeline, 
                                   param_distributions=param_distributions, 
                                   n_iter=20, 
                                   cv=3, 
                                   n_jobs=-1, 
                                   verbose=2, 
                                   random_state=42)

# Ajustar el modelo de búsqueda aleatoria
random_search.fit(X_train_imputed, y_train)

# Obtener el mejor modelo
best_model = random_search.best_estimator_

# Imprimir los mejores parámetros encontrados
print("Mejores parámetros encontrados: ", random_search.best_params_)

# Evaluar el mejor modelo en el conjunto de prueba
train_score = best_model.score(X_train_imputed, y_train)
print("Accuracy en el conjunto de entrenamiento: ", train_score)

# Hacer predicciones en el conjunto de prueba
y_pred = best_model.predict(X_test_imputed)

# Crear DataFrame con las predicciones y guardar en un archivo CSV
df_test_ids = pd.read_csv('test.csv')[['ID']] 
df_results = pd.DataFrame({'ID': df_test_ids['ID'], 'RENDIMIENTO_GLOBAL': y_pred})
df_results.to_csv('predicciones_decision_tree.csv', index=False)

print("Predicciones guardadas en 'predicciones_decision_tree.csv'")
